# Manual Observability (Spans & Instruments)

While `pydantic_ai` handles logging Agent and Tool execution out of the box automatically, you will often want to observe your *own* custom code, fallback logic, or database queries.

Logfire provides two extremely powerful concepts: **Spans** and **Instrumenting**.

In [2]:
import nest_asyncio
nest_asyncio.apply()

from dotenv import load_dotenv
load_dotenv()

import logfire
logfire.configure()

Logfire project URL: ]8;id=127426;https://logfire-us.pydantic.dev/d-hackmt/alia\https://logfire-us.pydantic.dev/d-hackmt/alia]8;;\

### 1. Manual `logfire.span(...)`

A "Span" is essentially a stopwatch. It logs when a block of code starts, how long it takes, and when it finishes (and if it threw an error!). You can use the `with` keyword to wrap any generic python code.

In [3]:
import time

# Manually opening a Span "stopwatch"
with logfire.span('Mock Database Connection') as span:
    logfire.info('Connecting to external DB...')
    time.sleep(1)  # Simulating a slow internet connection
    
    # We can inject data into the span halfway through!
    span.set_attribute('db_version', 'Postgres 15')
    
    logfire.info('Connected!')

13:51:25.882 Mock Database Connection
13:51:25.883   Connecting to external DB...
13:51:26.887   Connected!


### 2. Auto-Instrumenting Custom Functions

If you have a complex function and don't want to write `with logfire.span` every single time, you can just slap the `@logfire.instrument` decorator on top of it. 

Logfire will transparently capture the exact arguments sent to the function and the exact result returned!

In [4]:
@logfire.instrument('Processing Payment for {user_id}')
def process_payment(user_id: int, amount: float):
    # Logfire automatically intercepts `user_id` and tracks it!
    if amount > 1000:
        logfire.warn('Large transaction flagged for review.')
    return True

process_payment(user_id=8842, amount=150.00)
process_payment(user_id=8842, amount=9000.00)

13:51:27.858 Processing Payment for 8842
13:51:27.859 Processing Payment for 8842
13:51:27.860   Large transaction flagged for review.


True

On your Web Dashboard, you will now see perfectly nested spans containing these manual operations alongside your Pydantic AI calls!